<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 PAI-Bench Reproduction with Cosmos Framework

This notebook walks through generating the [PAI-Bench (Physical AI Bench) generation set](https://huggingface.co/datasets/shi-labs/physical-ai-bench-generation) with a Cosmos3 checkpoint (Cosmos3-Super by default, or Cosmos3-Nano) using the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

PAI-Bench covers Physical AI domains (AV driving, robotics, industry, physics, human, common sense) with **1044 samples**. This notebook runs **both** generation tasks:

- **Text-to-Video (T2V):** generate from the prompt only (no condition image).
- **Image-to-Video (I2V):** condition on the per-sample image (`condition_image/<image_name>`) and generate.

Both tasks use the local prompt assets under `assets/` (which carry `json_upsampled_prompt` and `negative_prompt`):

- `assets/t2v_prompts.json`
- `assets/i2v_prompts.json`

Generation target: **189 frames at 24 FPS**. We walk through one T2V demo and one I2V demo end-to-end, then provide two optional full-sweep cells (one per task).

## Reference Scores

Each task's **Overall Score** is the mean of a **Domain Score** (VQA / VLM-judge accuracy across the physical-AI domains) and a **Quality Score** (mean of the VBench quality dimensions): `Overall = (Domain + Quality) / 2` (computed in Section 12). The values below are reference numbers for each Cosmos3 variant.

| Variant | T2V Overall | T2V Domain | T2V Quality | I2V Overall | I2V Domain | I2V Quality |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Cosmos3-Super | 80.0% | 86.8% | 73.1% | 82.8% | 87.3% | 78.2% |
| Cosmos3-Nano | 79.4% | 85.8% | 73.0% | 82.7% | 87.2% | 78.1% |

## Prerequisites

- Linux machine with NVIDIA GPU access (default recipe uses 4 GPUs).
- Model access on Hugging Face. Either run `uvx hf@latest auth login` or set `HF_TOKEN` in the environment.
- `uv >= 0.11.3` installed (https://docs.astral.sh/uv/getting-started/installation/).
- `git-lfs` on PATH. The PAI-Bench dataset is downloaded by cloning the Hugging Face dataset repo (images are LFS-backed).
- Cache/output paths with enough disk space.

> **Headless servers:** if you see `libxcb.so.1: cannot open shared object file` when importing the model, install the system graphics libraries:
> ```bash
> apt-get install -y libxcb1 libgl1 libglib2.0-0
> ```

## 1. Configure Paths and Environment

All paths default to sensible locations under this `cosmos` checkout. Override any of them by exporting before launching the notebook:

```bash
export COSMOS3_REPO=/path/to/cosmos-framework
export COSMOS3_UV_GROUP=cu130-train   # or cu128-train
export UV_PROJECT_ENVIRONMENT=/path/to/large/uv/venvs/cosmos3-paibench
export COSMOS3_NUM_GPUS=4
export HF_HOME=/path/to/large/huggingface/cache
export CUDA_VISIBLE_DEVICES=0,1,2,3
export PAIBENCH_MODEL_VARIANT=Super   # or Nano (which Cosmos3 checkpoint to evaluate)
export PAIBENCH_DATASET_ROOT=/path/to/physical-ai-bench-generation
export PAIBENCH_OUTPUT_ROOT=/path/to/paibench/outputs
```

In [ ]:
from pathlib import Path
import os
import socket


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def free_local_port() -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return str(sock.getsockname()[1])


def default_framework_repo(root: Path) -> Path:
    for candidate in (root / "packages" / "cosmos-framework", root / "packages" / "cosmos3"):
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return root / "packages" / "cosmos-framework"


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS_ROOT))).resolve()
COSMOS3_GIT_URL = os.environ.get("COSMOS3_GIT_URL", "git@github.com:NVIDIA/cosmos-framework.git")
COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", "cu130-train")
COSMOS3_UV_ENV = Path(os.environ.get("UV_PROJECT_ENVIRONMENT", COSMOS3_REPO / ".venv")).resolve()
COSMOS3_NUM_GPUS = os.environ.get("COSMOS3_NUM_GPUS", "4")
CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1,2,3")

PAIBENCH_NOTEBOOK_ROOT = COSMOS_ROOT / "evaluation" / "cosmos3" / "generator" / "paibench_g"
PAIBENCH_ASSETS = PAIBENCH_NOTEBOOK_ROOT / "assets"
I2V_PROMPTS_FILE = PAIBENCH_ASSETS / "i2v_prompts.json"
T2V_PROMPTS_FILE = PAIBENCH_ASSETS / "t2v_prompts.json"

PAIBENCH_HF_URL = os.environ.get(
    "PAIBENCH_HF_URL", "https://huggingface.co/datasets/shi-labs/physical-ai-bench-generation"
)
PAIBENCH_DATASET_ROOT = Path(
    os.environ.get("PAIBENCH_DATASET_ROOT", PAIBENCH_NOTEBOOK_ROOT / "physical-ai-bench-generation")
).resolve()
PAIBENCH_CONDITION_IMAGE_DIR = PAIBENCH_DATASET_ROOT / "condition_image"
PAIBENCH_OUTPUT_ROOT = Path(
    os.environ.get("PAIBENCH_OUTPUT_ROOT", PAIBENCH_NOTEBOOK_ROOT / "outputs")
).resolve()
PAIBENCH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEMO_CASE_ID = os.environ.get("PAIBENCH_DEMO_CASE", "av_6a0c672b-9603-410e-bb4b-323c7c1b8029")
# Model variant to evaluate: "Super" or "Nano".
MODEL_VARIANT = os.environ.get("PAIBENCH_MODEL_VARIANT", "Super")
assert MODEL_VARIANT in ("Super", "Nano"), f"PAIBENCH_MODEL_VARIANT must be 'Super' or 'Nano', got {MODEL_VARIANT!r}"
CHECKPOINT = os.environ.get("PAIBENCH_CHECKPOINT", f"Cosmos3-{MODEL_VARIANT}")

MASTER_ADDR = os.environ.get("COSMOS3_MASTER_ADDR", "127.0.0.1")
MASTER_PORT_I2V = os.environ.get("COSMOS3_I2V_MASTER_PORT", free_local_port())
MASTER_PORT_T2V = os.environ.get("COSMOS3_T2V_MASTER_PORT", free_local_port())

for key, value in [
    ("COSMOS_ROOT", COSMOS_ROOT),
    ("COSMOS3_REPO", COSMOS3_REPO),
    ("COSMOS3_UV_ENV", COSMOS3_UV_ENV),
    ("COSMOS3_NUM_GPUS", COSMOS3_NUM_GPUS),
    ("CUDA_VISIBLE_DEVICES", CUDA_VISIBLE_DEVICES),
    ("PAIBENCH_ASSETS", PAIBENCH_ASSETS),
    ("PAIBENCH_DATASET_ROOT", PAIBENCH_DATASET_ROOT),
    ("PAIBENCH_OUTPUT_ROOT", PAIBENCH_OUTPUT_ROOT),
    ("DEMO_CASE_ID", DEMO_CASE_ID),
    ("MODEL_VARIANT", MODEL_VARIANT),
    ("CHECKPOINT", CHECKPOINT),
]:
    print(f"{key}={value}")

# Export for the %%bash cells below.
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_GIT_URL"] = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_UV_ENV"] = str(COSMOS3_UV_ENV)
os.environ["UV_PROJECT_ENVIRONMENT"] = str(COSMOS3_UV_ENV)
os.environ["COSMOS3_NUM_GPUS"] = COSMOS3_NUM_GPUS
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["PAIBENCH_HF_URL"] = PAIBENCH_HF_URL
os.environ["PAIBENCH_DATASET_ROOT"] = str(PAIBENCH_DATASET_ROOT)
os.environ["PAIBENCH_OUTPUT_ROOT"] = str(PAIBENCH_OUTPUT_ROOT)
os.environ["DEMO_CASE_ID"] = DEMO_CASE_ID
os.environ["CHECKPOINT"] = CHECKPOINT
os.environ["COSMOS3_MASTER_ADDR"] = MASTER_ADDR
os.environ["COSMOS3_I2V_MASTER_PORT"] = MASTER_PORT_I2V
os.environ["COSMOS3_T2V_MASTER_PORT"] = MASTER_PORT_T2V

## 2. Clone or Reuse Cosmos Framework

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -e "$COSMOS3_REPO" ]; then
  echo "COSMOS3_REPO exists but is not a Cosmos Framework checkout: $COSMOS3_REPO"
  exit 1
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

cd "$COSMOS3_REPO"
git status --short --branch
git remote -v

## 3. Install Native PyTorch Dependencies

Installs framework dependencies with the requested CUDA group (default `cu130-train`).

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

export GIT_LFS_SKIP_SMUDGE=1
cd "$COSMOS3_REPO"
export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_UV_ENV}"
echo "Using UV_PROJECT_ENVIRONMENT=$UV_PROJECT_ENVIRONMENT"
uv sync --all-extras --group="$COSMOS3_UV_GROUP"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "uv sync completed, but expected Python is missing: $COSMOS3_UV_ENV/bin/python"
  exit 1
fi

## 4. Verify GPU and Python Environment

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "Missing $COSMOS3_UV_ENV/bin/python"
  echo "Run the Install Native PyTorch Dependencies cell first."
  exit 1
fi
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" "$COSMOS3_UV_ENV/bin/python" - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"device {index}:", torch.cuda.get_device_name(index))
PY

## 5. Download the PAI-Bench Dataset

PAI-Bench is hosted on Hugging Face at [`shi-labs/physical-ai-bench-generation`](https://huggingface.co/datasets/shi-labs/physical-ai-bench-generation). We download it by cloning the dataset repo with Git LFS.

Layout (under `$PAIBENCH_DATASET_ROOT`):

```
physical-ai-bench-generation/
├── condition_image/                       # condition images (filenames match each sample's image_name)
├── vqa/                                   # VQA pairs (not used here)
└── cosmos_predict2_bench_full_info.json   # dataset metadata
```

> Only the condition images are read from the cloned dataset (for I2V). The prompts come from the **local** assets at `assets/i2v_prompts.json` and `assets/t2v_prompts.json` (which include `json_upsampled_prompt` and `negative_prompt`).

In [ ]:
%%bash
set -euo pipefail

if [ -d "$PAIBENCH_DATASET_ROOT/condition_image" ]; then
  echo "PAI-Bench dataset already present at $PAIBENCH_DATASET_ROOT"
  ls "$PAIBENCH_DATASET_ROOT"
  exit 0
fi

if ! command -v git-lfs >/dev/null 2>&1; then
  echo "git-lfs is required to download the PAI-Bench dataset (LFS-backed images)." >&2
  echo "Install it: https://git-lfs.com/  (e.g. apt-get install -y git-lfs)" >&2
  exit 1
fi

git lfs install
mkdir -p "$(dirname "$PAIBENCH_DATASET_ROOT")"
git clone "$PAIBENCH_HF_URL" "$PAIBENCH_DATASET_ROOT"

echo "--- contents of $PAIBENCH_DATASET_ROOT ---"
ls "$PAIBENCH_DATASET_ROOT"

## 6. Load Prompts and Preview the Demo Case

We load the T2V and I2V prompt assets, each keyed by `video_id` (1044 entries each). For I2V, the condition image for a case is `condition_image/<image_name>` inside the cloned dataset.

In [ ]:
import json
from IPython.display import Image, display

I2V_PROMPTS = {row["video_id"]: row for row in json.loads(I2V_PROMPTS_FILE.read_text())}
T2V_PROMPTS = {row["video_id"]: row for row in json.loads(T2V_PROMPTS_FILE.read_text())}

print(f"Loaded {len(I2V_PROMPTS)} I2V prompts from {I2V_PROMPTS_FILE.name}")
print(f"Loaded {len(T2V_PROMPTS)} T2V prompts from {T2V_PROMPTS_FILE.name}")
assert len(I2V_PROMPTS) == 1044, f"expected 1044 I2V prompts, got {len(I2V_PROMPTS)}"
assert len(T2V_PROMPTS) == 1044, f"expected 1044 T2V prompts, got {len(T2V_PROMPTS)}"

assert DEMO_CASE_ID in I2V_PROMPTS, f"{DEMO_CASE_ID} not found in I2V prompts"
assert DEMO_CASE_ID in T2V_PROMPTS, f"{DEMO_CASE_ID} not found in T2V prompts"

entry = I2V_PROMPTS[DEMO_CASE_ID]
demo_image = PAIBENCH_CONDITION_IMAGE_DIR / entry["image_name"]
print("\ndemo case:", DEMO_CASE_ID)
print("condition image (I2V):", demo_image)
print("prompt_en:", entry.get("prompt_en", "")[:300], "...")

if demo_image.exists():
    display(Image(filename=str(demo_image), width=480))
else:
    print("Condition image not found yet - run the dataset download cell first.")

## 7. Helper Functions

**Recipe (shared by T2V and I2V)**

- **Output length**: 189 frames at 24 fps.
- **Sampling**: `num_steps=50`, `guidance=6.0`, `shift=10.0`, `seed=0` (all other sampler settings left at framework defaults).
- **Positive prompt**: the per-case `json_upsampled_prompt` string, passed through verbatim.
- **Negative prompt**: the shared `negative_prompt` string, passed through verbatim.
- **T2V**: `model_mode="text2video"`, no condition image.
- **I2V**: `model_mode="image2video"`, conditioned on `condition_image/<image_name>`.

Helpers:

- `case_image_path(case_id)` — resolve the I2V condition image (falls back to a recursive search if the clone nests the images).
- `build_t2v_row(case_id)` / `build_i2v_row(case_id)` — assemble inference JSONL rows.
- `build_input_jsonl(rows, dst)` — write one or more rows to a JSONL file.

In [ ]:
NUM_FRAMES = 189
FPS = 24
RESOLUTION = "720"
ASPECT_RATIO = "16,9"
NUM_STEPS = 50
GUIDANCE = 6.0
SHIFT = 10.0
SEED = 0


def case_image_path(case_id: str) -> Path:
    entry = I2V_PROMPTS[case_id]
    image_name = entry["image_name"]
    direct = PAIBENCH_CONDITION_IMAGE_DIR / image_name
    if direct.exists():
        return direct
    # Fall back to a recursive search in case the clone nests the images differently.
    matches = sorted(PAIBENCH_DATASET_ROOT.rglob(image_name))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"condition image not found for {case_id}: {direct}")


def build_t2v_row(case_id: str) -> dict:
    entry = T2V_PROMPTS[case_id]
    return {
        "aspect_ratio": ASPECT_RATIO,
        "fps": FPS,
        "guidance": GUIDANCE,
        "model_mode": "text2video",
        "name": case_id,
        "negative_prompt": entry["negative_prompt"],
        "num_frames": NUM_FRAMES,
        "num_outputs": 1,
        "num_steps": NUM_STEPS,
        "prompt": entry["json_upsampled_prompt"],
        "resolution": RESOLUTION,
        "seed": SEED,
        "shift": SHIFT,
    }


def build_i2v_row(case_id: str) -> dict:
    entry = I2V_PROMPTS[case_id]
    return {
        "aspect_ratio": ASPECT_RATIO,
        "fps": FPS,
        "guidance": GUIDANCE,
        "model_mode": "image2video",
        "name": case_id,
        "negative_prompt": entry["negative_prompt"],
        "num_frames": NUM_FRAMES,
        "num_outputs": 1,
        "num_steps": NUM_STEPS,
        "prompt": entry["json_upsampled_prompt"],
        "resolution": RESOLUTION,
        "seed": SEED,
        "shift": SHIFT,
        "vision_path": str(case_image_path(case_id)),
    }


def build_input_jsonl(rows: list[dict], dst_jsonl: Path) -> Path:
    dst_jsonl.parent.mkdir(parents=True, exist_ok=True)
    with dst_jsonl.open("w") as fp:
        for row in rows:
            fp.write(json.dumps(row) + "\n")
    return dst_jsonl


def display_video(path: Path, width: int = 480) -> None:
    from IPython.display import Video, display
    display(Video(filename=str(path), embed=True, width=width))

## 8. I2V Demo — Build, Run, Preview

In [ ]:
i2v_run_dir = PAIBENCH_OUTPUT_ROOT / "i2v" / DEMO_CASE_ID
i2v_run_dir.mkdir(parents=True, exist_ok=True)

i2v_input_jsonl = i2v_run_dir / "input.jsonl"
i2v_output_dir = i2v_run_dir / "raw"
i2v_output_dir.mkdir(parents=True, exist_ok=True)

build_input_jsonl([build_i2v_row(DEMO_CASE_ID)], i2v_input_jsonl)

os.environ["I2V_INPUT"] = str(i2v_input_jsonl)
os.environ["I2V_OUTPUT_DIR"] = str(i2v_output_dir)

print("I2V_INPUT =", i2v_input_jsonl)
print("I2V_OUTPUT_DIR =", i2v_output_dir)
print()
print("row preview:")
print(i2v_input_jsonl.read_text()[:600], "...")

### Run I2V Inference

We use the **latency** parallelism preset with context-parallel sharding across all visible GPUs (`--cp-size=$COSMOS3_NUM_GPUS`).

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_INPUT" \
  -o "$I2V_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

In [ ]:
raw_outputs = sorted(i2v_output_dir.rglob("vision.mp4"))
if raw_outputs:
    raw_mp4 = raw_outputs[0]
    print("raw I2V output:", raw_mp4)
    display_video(raw_mp4)
else:
    print("No vision.mp4 found yet - run the I2V inference cell first.")

## 9. T2V Demo — Build, Run, Preview

Text-to-Video uses the same case prompt but no condition image.

In [ ]:
t2v_run_dir = PAIBENCH_OUTPUT_ROOT / "t2v" / DEMO_CASE_ID
t2v_run_dir.mkdir(parents=True, exist_ok=True)

t2v_input_jsonl = t2v_run_dir / "input.jsonl"
t2v_output_dir = t2v_run_dir / "raw"
t2v_output_dir.mkdir(parents=True, exist_ok=True)

build_input_jsonl([build_t2v_row(DEMO_CASE_ID)], t2v_input_jsonl)

os.environ["T2V_INPUT"] = str(t2v_input_jsonl)
os.environ["T2V_OUTPUT_DIR"] = str(t2v_output_dir)

print("T2V_INPUT =", t2v_input_jsonl)
print("T2V_OUTPUT_DIR =", t2v_output_dir)
print()
print("row preview:")
print(t2v_input_jsonl.read_text()[:600], "...")

### Run T2V Inference

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_T2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$T2V_INPUT" \
  -o "$T2V_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

In [ ]:
raw_outputs = sorted(t2v_output_dir.rglob("vision.mp4"))
if raw_outputs:
    raw_mp4 = raw_outputs[0]
    print("raw T2V output:", raw_mp4)
    display_video(raw_mp4)
else:
    print("No vision.mp4 found yet - run the T2V inference cell first.")

## 10. (Optional) Run All 1044 Cases — I2V

Set `RUN_ALL_I2V = True` to write a single combined JSONL covering every I2V case, then run the following bash cell to generate them all.

In [ ]:
RUN_ALL_I2V = False  # set to True to enable the full 1044-case I2V sweep

if RUN_ALL_I2V:
    case_ids = sorted(I2V_PROMPTS.keys())
    assert len(case_ids) == 1044

    i2v_inputs_dir = PAIBENCH_OUTPUT_ROOT / "i2v_full" / "inputs"
    i2v_raw_dir = PAIBENCH_OUTPUT_ROOT / "i2v_full" / "raw"
    i2v_inputs_dir.mkdir(parents=True, exist_ok=True)
    i2v_raw_dir.mkdir(parents=True, exist_ok=True)

    all_jsonl = i2v_inputs_dir / "all_1044.jsonl"
    build_input_jsonl([build_i2v_row(case_id) for case_id in case_ids], all_jsonl)

    os.environ["I2V_FULL_INPUT"] = str(all_jsonl)
    os.environ["I2V_FULL_OUTPUT_DIR"] = str(i2v_raw_dir)
    print("I2V_FULL_INPUT =", all_jsonl)
    print("I2V_FULL_OUTPUT_DIR =", i2v_raw_dir)
    print(f"Wrote {len(case_ids)} rows. Run the next bash cell to generate all I2V outputs.")
else:
    print("Set RUN_ALL_I2V = True above and re-run to enable the full 1044-case I2V sweep.")

### Generate All I2V Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${I2V_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL_I2V = True in the previous Python cell and re-run it first."
  exit 0
fi

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_FULL_INPUT" \
  -o "$I2V_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

## 11. (Optional) Run All 1044 Cases — T2V

Set `RUN_ALL_T2V = True` to write a single combined JSONL covering every T2V case, then run the following bash cell to generate them all.

In [ ]:
RUN_ALL_T2V = False  # set to True to enable the full 1044-case T2V sweep

if RUN_ALL_T2V:
    case_ids = sorted(T2V_PROMPTS.keys())
    assert len(case_ids) == 1044

    t2v_inputs_dir = PAIBENCH_OUTPUT_ROOT / "t2v_full" / "inputs"
    t2v_raw_dir = PAIBENCH_OUTPUT_ROOT / "t2v_full" / "raw"
    t2v_inputs_dir.mkdir(parents=True, exist_ok=True)
    t2v_raw_dir.mkdir(parents=True, exist_ok=True)

    all_jsonl = t2v_inputs_dir / "all_1044.jsonl"
    build_input_jsonl([build_t2v_row(case_id) for case_id in case_ids], all_jsonl)

    os.environ["T2V_FULL_INPUT"] = str(all_jsonl)
    os.environ["T2V_FULL_OUTPUT_DIR"] = str(t2v_raw_dir)
    print("T2V_FULL_INPUT =", all_jsonl)
    print("T2V_FULL_OUTPUT_DIR =", t2v_raw_dir)
    print(f"Wrote {len(case_ids)} rows. Run the next bash cell to generate all T2V outputs.")
else:
    print("Set RUN_ALL_T2V = True above and re-run to enable the full 1044-case T2V sweep.")

### Generate All T2V Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${T2V_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL_T2V = True in the previous Python cell and re-run it first."
  exit 0
fi

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_T2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$T2V_FULL_INPUT" \
  -o "$T2V_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

## 12. (Optional) Score the Generated Videos — PAI-Bench-G

This section scores the generated videos with the official [PAI-Bench-G scorer](https://github.com/SHI-Labs/physical-ai-bench) (`generation/`). The PAI-Bench-G score combines two evaluations:

1. **Video quality** (VBench-based, `evaluate.py`) — per-dimension scores in `[0, 1]`:
   - **T2V (6 dims):** `aesthetic_quality`, `background_consistency`, `imaging_quality`, `motion_smoothness`, `overall_consistency`, `subject_consistency`.
   - **I2V (8 dims):** the same 6 plus `i2v_background`, `i2v_subject` (DreamSim-based).
2. **VLM judge / VQA** (`evaluate_vqa.py`) — `Qwen/Qwen2.5-VL-72B-Instruct` via vLLM answers per-video questions; reports `overall_accuracy` in `[0, 1]`.

The final score per task is:

```
PAI-Bench-G = (mean(quality dims) + VQA overall_accuracy) / 2 * 100
```

Reference numbers for Cosmos3-Super: **T2V 80.0%**, **I2V 82.8%**.

**Requirements**

- Generate the full sweeps first (Section 10 for I2V, Section 11 for T2V, `RUN_ALL_* = True`). The aggregate is only meaningful over the full 1044-case sets; a demo case works as a smoke test.
- ~4 GPUs. VQA loads Qwen2.5-VL-72B via vLLM (tensor-parallel); quality runs across the GPUs with `torch.distributed.run`.
- Extra disk for the scorer env, VBench/DreamSim model weights, and Qwen weights (~140 GB on first download unless a local snapshot is provided).

### 12.1 Configure Scoring Paths

Sets the scorer location, which tasks to score, the quality dimensions per task, and the VQA model. Override any of these by exporting before launching the notebook:

```bash
export PAIBENCH_SCORER_REPO=/path/to/physical-ai-bench
export PAIBENCH_SCORER_GIT_URL=https://github.com/SHI-Labs/physical-ai-bench.git
export PAIBENCH_SCORER_COMMIT=            # optional: pin a specific scorer commit
export QWEN_MODEL_PATH=Qwen/Qwen2.5-VL-72B-Instruct   # HF id or local snapshot path
export PAIBENCH_SCORE_TASKS="t2v i2v"     # subset to "t2v" or "i2v" if you only need one
```

In [ ]:
RUN_SCORING = False  # set to True to enable PAI-Bench-G scoring

# Which tasks to score and the quality dimensions each uses.
PAIBENCH_SCORE_TASKS = os.environ.get("PAIBENCH_SCORE_TASKS", "t2v i2v").split()
QUALITY_DIMS_T2V = [
    "aesthetic_quality",
    "background_consistency",
    "imaging_quality",
    "motion_smoothness",
    "overall_consistency",
    "subject_consistency",
]
QUALITY_DIMS_I2V = QUALITY_DIMS_T2V + ["i2v_background", "i2v_subject"]

PAIBENCH_SCORER_REPO = Path(
    os.environ.get("PAIBENCH_SCORER_REPO", PAIBENCH_NOTEBOOK_ROOT / "scorers" / "physical-ai-bench")
).resolve()
PAIBENCH_SCORER_DIR = PAIBENCH_SCORER_REPO / "generation"
PAIBENCH_SCORER_GIT_URL = os.environ.get(
    "PAIBENCH_SCORER_GIT_URL", "https://github.com/SHI-Labs/physical-ai-bench.git"
)
PAIBENCH_SCORER_COMMIT = os.environ.get("PAIBENCH_SCORER_COMMIT", "")

QWEN_MODEL_PATH = os.environ.get("QWEN_MODEL_PATH", "Qwen/Qwen2.5-VL-72B-Instruct")

PAIBENCH_PROMPT_FILE = PAIBENCH_DATASET_ROOT / "cosmos_predict2_bench_full_info.json"
PAIBENCH_VQA_DIR = PAIBENCH_DATASET_ROOT / "vqa"
PAIBENCH_SCORES_ROOT = PAIBENCH_OUTPUT_ROOT / "scores"

# HF cache for Qwen weights (kept off the home partition by default).
PAIBENCH_SCORE_HF_HOME = Path(
    os.environ.get("PAIBENCH_SCORE_HF_HOME", PAIBENCH_SCORER_REPO / ".hf_cache")
).resolve()

for key, value in [
    ("RUN_SCORING", RUN_SCORING),
    ("PAIBENCH_SCORE_TASKS", PAIBENCH_SCORE_TASKS),
    ("PAIBENCH_SCORER_REPO", PAIBENCH_SCORER_REPO),
    ("QWEN_MODEL_PATH", QWEN_MODEL_PATH),
    ("PAIBENCH_PROMPT_FILE", PAIBENCH_PROMPT_FILE),
    ("PAIBENCH_VQA_DIR", PAIBENCH_VQA_DIR),
    ("PAIBENCH_SCORES_ROOT", PAIBENCH_SCORES_ROOT),
]:
    print(f"{key}={value}")

os.environ["RUN_SCORING"] = str(RUN_SCORING)
os.environ["PAIBENCH_SCORER_REPO"] = str(PAIBENCH_SCORER_REPO)
os.environ["PAIBENCH_SCORER_DIR"] = str(PAIBENCH_SCORER_DIR)
os.environ["PAIBENCH_SCORER_GIT_URL"] = PAIBENCH_SCORER_GIT_URL
os.environ["PAIBENCH_SCORER_COMMIT"] = PAIBENCH_SCORER_COMMIT
os.environ["QWEN_MODEL_PATH"] = QWEN_MODEL_PATH
os.environ["PAIBENCH_PROMPT_FILE"] = str(PAIBENCH_PROMPT_FILE)
os.environ["PAIBENCH_VQA_DIR"] = str(PAIBENCH_VQA_DIR)
os.environ["PAIBENCH_SCORES_ROOT"] = str(PAIBENCH_SCORES_ROOT)
os.environ["PAIBENCH_SCORE_TASKS"] = " ".join(PAIBENCH_SCORE_TASKS)
os.environ["PAIBENCH_QUALITY_DIMS_T2V"] = " ".join(QUALITY_DIMS_T2V)
os.environ["PAIBENCH_QUALITY_DIMS_I2V"] = " ".join(QUALITY_DIMS_I2V)
os.environ["PAIBENCH_SCORE_HF_HOME"] = str(PAIBENCH_SCORE_HF_HOME)

if not RUN_SCORING:
    print("\nSet RUN_SCORING = True above and re-run to enable PAI-Bench-G scoring.")

### 12.2 Clone the PAI-Bench-G Scorer

Clones (or reuses) the scorer repo. Optionally pin `PAIBENCH_SCORER_COMMIT` for an exact reproduction.

In [ ]:
%%bash
set -euo pipefail

if [ -d "$PAIBENCH_SCORER_DIR/pbench" ]; then
  echo "Using existing PAI-Bench scorer checkout: $PAIBENCH_SCORER_REPO"
else
  echo "Cloning $PAIBENCH_SCORER_GIT_URL into $PAIBENCH_SCORER_REPO"
  mkdir -p "$(dirname "$PAIBENCH_SCORER_REPO")"
  git clone "$PAIBENCH_SCORER_GIT_URL" "$PAIBENCH_SCORER_REPO"
fi

cd "$PAIBENCH_SCORER_REPO"
if [ -n "${PAIBENCH_SCORER_COMMIT:-}" ]; then
  echo "Checking out pinned commit: $PAIBENCH_SCORER_COMMIT"
  git checkout "$PAIBENCH_SCORER_COMMIT"
fi
git rev-parse --short HEAD
ls generation

### 12.3 Set up the Scorer Environment

The scorer is its own `uv` project under `generation/`. This creates its dedicated env (`generation/.venv`, separate from the generation framework env) and installs `detectron2` without build isolation, per the scorer's setup. One-time; re-running is a no-op once synced.

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

cd "$PAIBENCH_SCORER_DIR"
uv sync
uv pip install --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"

echo "--- scorer env sanity ---"
uv run python - <<'PY'
import torch, vllm  # noqa: F401
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("vllm:", vllm.__version__)
PY

### 12.4 Organize Generated Videos

The scorer expects a flat directory of `{video_id}__{seed}.mp4` files. This symlinks each generated `vision.mp4` into a per-task `organized/` folder as `{video_id}__0.mp4`. By default it scores the full sweeps (`t2v_full/raw`, `i2v_full/raw`); override `PAIBENCH_<TASK>_RAW_DIR` to score a demo raw dir instead.

In [ ]:
if not RUN_SCORING:
    print("RUN_SCORING is False - set it to True in the config cell (12.1) and re-run.")
else:
    def organize_task(task: str) -> Path:
        raw_dir = Path(
            os.environ.get(f"PAIBENCH_{task.upper()}_RAW_DIR", PAIBENCH_OUTPUT_ROOT / f"{task}_full" / "raw")
        ).resolve()
        organized_dir = (PAIBENCH_OUTPUT_ROOT / f"{task}_full" / "organized").resolve()
        organized_dir.mkdir(parents=True, exist_ok=True)

        if not raw_dir.exists():
            print(f"  [{task}] no raw outputs at {raw_dir} - generate first or set PAIBENCH_{task.upper()}_RAW_DIR")
            return organized_dir

        linked = 0
        for mp4 in sorted(raw_dir.rglob("vision.mp4")):
            video_id = mp4.parent.name.split("__")[0]
            dst = organized_dir / f"{video_id}__0.mp4"
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            dst.symlink_to(mp4.resolve())
            linked += 1
        print(f"  [{task}] linked {linked} videos -> {organized_dir}")
        os.environ[f"PAIBENCH_{task.upper()}_VIDEOS"] = str(organized_dir)
        return organized_dir

    for task in PAIBENCH_SCORE_TASKS:
        organize_task(task)

### 12.5 Run Video Quality Evaluation

Runs `evaluate.py` across the GPUs with `torch.distributed.run`, scoring the per-task dimensions (6 for T2V, 8 for I2V). Results are written under `scores/<task>/quality/` as timestamped `*_eval_results.json`. First run downloads the VBench / DreamSim / DINO weights into the caches below.

> **Troubleshooting (`i2v_background` / `i2v_subject`):** these DreamSim dims can fail with a torchao version error from `peft` if the installed `torchao < 0.16`. If you hit that, upgrade torchao in the scorer env (`uv pip install --python "$PAIBENCH_SCORER_DIR/.venv/bin/python" -U torchao`) or drop those two dims from the I2V run.

In [ ]:
%%bash
set -euo pipefail

if [ "${RUN_SCORING:-False}" != "True" ]; then
  echo "RUN_SCORING is not True. Set RUN_SCORING = True in cell 12.1 and run the prep cells first."
  exit 0
fi

cd "$PAIBENCH_SCORER_DIR"
export HF_HOME="$PAIBENCH_SCORE_HF_HOME"
export VBENCH_CACHE_DIR="$PAIBENCH_SCORER_REPO/.vbench_cache"
mkdir -p "$HF_HOME" "$VBENCH_CACHE_DIR"

for TASK in $PAIBENCH_SCORE_TASKS; do
  VIDEOS_DIR="$PAIBENCH_OUTPUT_ROOT/${TASK}_full/organized"
  RESULTS_DIR="$PAIBENCH_SCORES_ROOT/$TASK/quality"

  if [ ! -d "$VIDEOS_DIR" ] || [ -z "$(ls -A "$VIDEOS_DIR" 2>/dev/null)" ]; then
    echo "--- [skip] $TASK: no organized videos at $VIDEOS_DIR ---"
    continue
  fi

  if [ "$TASK" = "i2v" ]; then
    DIMS="$PAIBENCH_QUALITY_DIMS_I2V"
  else
    DIMS="$PAIBENCH_QUALITY_DIMS_T2V"
  fi

  echo "--- Quality scoring: $TASK (dims: $DIMS) ---"
  mkdir -p "$RESULTS_DIR"
  CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
  uv run python -m torch.distributed.run --standalone --nproc_per_node "$COSMOS3_NUM_GPUS" evaluate.py \
    --mode custom_input \
    --prompt_file "$PAIBENCH_PROMPT_FILE" \
    --custom_image_folder "$PAIBENCH_DATASET_ROOT/condition_image" \
    --dimension $DIMS \
    --videos_path "$VIDEOS_DIR" \
    --output_path "$RESULTS_DIR/" \
    --enable_missing_videos
  echo "--- Quality done: $TASK ---"
done

### 12.6 Run VLM Judge (VQA) Evaluation

Runs `evaluate_vqa.py` with `Qwen/Qwen2.5-VL-72B-Instruct` via vLLM (tensor-parallel across the visible GPUs), writing `vqa_summary.json` (with `overall_accuracy`) under `scores/<task>/vqa/`. First run downloads the Qwen weights into the HF cache unless `QWEN_MODEL_PATH` points at a local snapshot.

In [ ]:
%%bash
set -euo pipefail

if [ "${RUN_SCORING:-False}" != "True" ]; then
  echo "RUN_SCORING is not True. Set RUN_SCORING = True in cell 12.1 and run the prep cells first."
  exit 0
fi

cd "$PAIBENCH_SCORER_DIR"
export HF_HOME="$PAIBENCH_SCORE_HF_HOME"
mkdir -p "$HF_HOME"

for TASK in $PAIBENCH_SCORE_TASKS; do
  VIDEOS_DIR="$PAIBENCH_OUTPUT_ROOT/${TASK}_full/organized"
  RESULTS_DIR="$PAIBENCH_SCORES_ROOT/$TASK/vqa"

  if [ ! -d "$VIDEOS_DIR" ] || [ -z "$(ls -A "$VIDEOS_DIR" 2>/dev/null)" ]; then
    echo "--- [skip] $TASK: no organized videos at $VIDEOS_DIR ---"
    continue
  fi

  echo "--- VQA scoring: $TASK ---"
  mkdir -p "$RESULTS_DIR"
  CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" \
  uv run python evaluate_vqa.py \
    --model_name "$QWEN_MODEL_PATH" \
    --tensor_parallel_size "$COSMOS3_NUM_GPUS" \
    --prompt_file "$PAIBENCH_PROMPT_FILE" \
    --vqa_questions_dir "$PAIBENCH_VQA_DIR" \
    --video_dir "$VIDEOS_DIR" \
    --output_dir "$RESULTS_DIR" \
    --enable_missing_videos
  echo "--- VQA done: $TASK ---"
done

### 12.7 Compute the Final PAI-Bench-G Score

Reads the quality `*_eval_results.json` and the VQA `vqa_summary.json` for each task, then reports per-dimension scores, the quality mean, the VQA accuracy, and the combined score:

```
PAI-Bench-G = (mean(quality dims) + VQA overall_accuracy) / 2 * 100
```

Reference numbers for Cosmos3-Super: T2V 80.0%, I2V 82.8%.

In [ ]:
TARGETS = {"t2v": 80.0, "i2v": 82.8}  # Cosmos3-Super reference numbers
is_super = "super" in CHECKPOINT.lower()


def load_quality_scores(results_dir: Path, dims: list[str]) -> dict[str, float]:
    """Merge VBench-style *_eval_results.json files into {dim: score}."""
    scores: dict[str, float] = {}
    for jf in sorted(results_dir.glob("*_eval_results.json")):
        data = json.loads(jf.read_text())
        for dim, val in data.items():
            # VBench stores {dim: [aggregate_score, [per-video...]]}; sometimes a bare float.
            score = val[0] if isinstance(val, (list, tuple)) else val
            if dim in dims:
                scores[dim] = float(score)
    return scores


for task in PAIBENCH_SCORE_TASKS:
    dims = QUALITY_DIMS_I2V if task == "i2v" else QUALITY_DIMS_T2V
    quality_dir = PAIBENCH_SCORES_ROOT / task / "quality"
    vqa_summary = PAIBENCH_SCORES_ROOT / task / "vqa" / "vqa_summary.json"

    print(f"\n=== PAI-Bench-G {task.upper()} ({CHECKPOINT}) ===")

    quality = load_quality_scores(quality_dir, dims) if quality_dir.exists() else {}
    for dim in dims:
        if dim in quality:
            print(f"  {dim:<24} {quality[dim]:.4f}")
        else:
            print(f"  {dim:<24} (missing)")

    if len(quality) != len(dims):
        print("  [incomplete] not all quality dims present - run cell 12.5 first.")
        continue
    quality_mean = sum(quality.values()) / len(dims)
    print(f"  {'quality mean':<24} {quality_mean:.4f}")

    if not vqa_summary.exists():
        print("  [incomplete] vqa_summary.json missing - run cell 12.6 first.")
        continue
    vqa_acc = float(json.loads(vqa_summary.read_text())["overall_accuracy"])
    print(f"  {'VQA overall_accuracy':<24} {vqa_acc:.4f}")

    final = (quality_mean + vqa_acc) / 2 * 100
    line = f"  {'PAI-Bench-G ' + task.upper():<24} {final:.2f}%"
    if is_super and task in TARGETS:
        line += f"   (reference {TARGETS[task]:.1f}%)"
    print(line)